In [0]:
dbutils.widgets.text("catalog", "shopstream_dev")
catalog = dbutils.widgets.get("catalog")
events = f"/Volumes/{catalog}/bronze/events/orders_stream"
checkpoints = f"/Volumes/{catalog}/bronze/checkpoints/order_events"

(spark.readStream.format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("cloudFiles.schemaLocation", f"{checkpoints}/schema")
   .load(events)
   .selectExpr("*", "_metadata.file_path AS _source_file", "current_timestamp() AS _ingested_at")
   .writeStream
   .option("checkpointLocation", f"{checkpoints}/stream")
   .trigger(availableNow=True)
   .toTable(f"{catalog}.bronze.order_events")
   .awaitTermination())

display(spark.sql(f"SELECT COUNT(*) AS events FROM {catalog}.bronze.order_events"))